# 第6回: Mamba を用いた時系列学習

このノートブックでは、**Mamba (Selective State Space Model)** を用いた時系列予測について学びます。
Mambaは2023年に提案された新しいアーキテクチャで、Transformerに匹敵する性能を
O(N)の計算量で実現します。

**学習内容:**
1. **State Space Models (SSM)** の基本概念
2. **Mambaの特徴** --- 選択的SSMのメカニズム
3. **モデル実装** --- MambaBlock の構造と実装
4. **Robomimicデータでの学習** --- 関節角度予測とLSTMとの比較

## 環境準備

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

---
## 1. State Space Models (SSM) の基本概念

### 連続時間 State Space Model

State Space Model (SSM) は、制御理論に由来する時系列モデルで、
隠れ状態 $x(t) \in \mathbb{R}^N$ を介して入力 $u(t)$ を出力 $y(t)$ に変換します。

$$
x'(t) = A\,x(t) + B\,u(t)
$$
$$
y(t) = C\,x(t)
$$

ここで：
- $A \in \mathbb{R}^{N \times N}$ : 状態遷移行列
- $B \in \mathbb{R}^{N \times 1}$ : 入力行列
- $C \in \mathbb{R}^{1 \times N}$ : 出力行列

### 離散化

ニューラルネットワークで扱うために、連続時間SSMをタイムステップ $\Delta$ で離散化します：

$$
x_k = \bar{A}\, x_{k-1} + \bar{B}\, u_k
$$
$$
y_k = C\, x_k
$$

ここで $\bar{A} = \exp(\Delta A)$、$\bar{B} = (\Delta A)^{-1}(\exp(\Delta A) - I) \cdot \Delta B$ です。

### Mambaの革新：選択的SSM

従来のSSMでは $A, B, C$ は入力に依存しない固定パラメータでしたが、
**Mamba** ではこれらを**入力依存**にすることで、文脈に応じた選択的な情報処理を実現します：

- $B, C, \Delta$ が入力 $x$ に依存して動的に変化
- 入力に応じて「どの情報を記憶し、どの情報を忘れるか」を選択可能

### RNN・Transformer との比較

| 特性 | RNN (LSTM) | Transformer | Mamba |
|------|-----------|-------------|-------|
| 計算量 | O(N) | O(N^2) | O(N) |
| 並列学習 | 不可（逐次処理） | 可能 | 可能（スキャン演算） |
| 長期依存性 | 勾配消失の問題 | 注意機構で対応 | SSMの構造で対応 |
| メモリ | O(1) 固定長 | O(N) KVキャッシュ | O(1) 固定長 |
| 推論速度 | 速い（逐次） | 遅い（全系列参照） | 速い（逐次） |
| 入力選択性 | ゲート機構 | Attention重み | 選択的SSMパラメータ |

**Mambaの利点:**
- Transformerのような並列学習が可能でありながら、RNNのような効率的な推論が可能
- 系列長に対して線形の計算量で、長い系列でもスケーラブル
- 入力依存パラメータにより、文脈に応じた柔軟な情報処理が可能

---
## 2. モデル実装

### 2.1 SelectiveSSM

Mambaの中核は**選択的SSM (Selective State Space Model)** です。
各タイムステップで入力に応じて $B, C, \Delta$ を動的に計算し、
状態を逐次更新します。

In [ ]:
class SelectiveSSM(nn.Module):
    """簡略化した選択的State Space Model（教育用実装）"""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # SSMパラメータ A（対数スケールで保持し、負の値を保証）
        self.A_log = nn.Parameter(torch.randn(d_model, d_state))

        # 入力依存の射影
        self.proj_B = nn.Linear(d_model, d_state)    # B: 入力→状態
        self.proj_C = nn.Linear(d_model, d_state)    # C: 状態→出力
        self.proj_delta = nn.Linear(d_model, d_model) # delta: 離散化ステップ

        # スキップ接続パラメータ
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            y: (batch, seq_len, d_model)
        """
        batch, seq_len, d_model = x.shape

        # 入力依存パラメータの計算
        B = self.proj_B(x)                              # (batch, seq_len, d_state)
        C = self.proj_C(x)                              # (batch, seq_len, d_state)
        delta = torch.softplus(self.proj_delta(x))      # (batch, seq_len, d_model)

        # Aを負の値として復元（安定性のため）
        A = -torch.exp(self.A_log)                      # (d_model, d_state)

        # 逐次スキャン
        y = torch.zeros_like(x)
        h = torch.zeros(batch, d_model, self.d_state, device=x.device)

        for t in range(seq_len):
            # 離散化: dA = exp(delta * A)
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A.unsqueeze(0))
            # (batch, d_model, d_state)

            # 離散化: dB = delta * B
            dB = delta[:, t].unsqueeze(-1) * B[:, t].unsqueeze(1)
            # (batch, d_model, d_state)

            # 状態更新: h_t = dA * h_{t-1} + dB * x_t
            h = dA * h + dB * x[:, t].unsqueeze(-1)

            # 出力: y_t = C_t * h_t + D * x_t
            y[:, t] = (h * C[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t]

        return y


# 動作確認
ssm = SelectiveSSM(d_model=32, d_state=16)
test_input = torch.randn(2, 10, 32)
test_output = ssm(test_input)
print(f"入力形状:  {test_input.shape}")
print(f"出力形状:  {test_output.shape}")
print(f"パラメータ数: {sum(p.numel() for p in ssm.parameters()):,}")

### 2.2 MambaBlock

MambaBlockは以下の構成要素を組み合わせた完全なブロックです：

1. **LayerNorm** --- 入力の正規化
2. **Linear射影** --- チャンネル拡張（expand倍）+ ゲート用の分岐
3. **Conv1d** --- 局所的な文脈の抽出（因果的畳み込み）
4. **SelectiveSSM** --- 選択的状態空間モデルによる系列処理
5. **ゲート機構** --- SiLU活性化によるゲーティング
6. **残差接続** --- 学習の安定化

In [ ]:
class MambaBlock(nn.Module):
    """Mambaブロック"""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner = d_model * expand

        self.in_proj = nn.Linear(d_model, d_inner * 2)  # x と z に分岐
        self.conv1d = nn.Conv1d(
            d_inner, d_inner, d_conv,
            padding=d_conv - 1, groups=d_inner  # depthwise conv
        )
        self.ssm = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)

        # 2つのパスに分岐
        xz = self.in_proj(x)
        x, z = xz.chunk(2, dim=-1)

        # 因果的Conv1d（局所的な文脈抽出）
        x = x.transpose(1, 2)                        # (batch, d_inner, seq_len)
        x = self.conv1d(x)[:, :, :x.shape[-1]]       # パディング分をカット
        x = x.transpose(1, 2)                        # (batch, seq_len, d_inner)
        x = torch.silu(x)

        # SelectiveSSM
        x = self.ssm(x)

        # ゲート機構: SSM出力 * SiLU(z)
        x = x * torch.silu(z)
        x = self.out_proj(x)

        # 残差接続
        return x + residual


# 動作確認
block = MambaBlock(d_model=32, d_state=16)
test_input = torch.randn(2, 10, 32)
test_output = block(test_input)
print(f"入力形状:  {test_input.shape}")
print(f"出力形状:  {test_output.shape}")
print(f"パラメータ数: {sum(p.numel() for p in block.parameters()):,}")

### 2.3 MambaModel

複数のMambaBlockを積み重ねた完全なモデルです。
入力射影 → MambaBlock x N → 出力射影 の構成になります。

In [ ]:
class MambaModel(nn.Module):
    """Mambaによる時系列予測モデル"""
    def __init__(self, input_dim, d_model=64, d_state=16, n_layers=2, output_dim=None):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim

        self.input_proj = nn.Linear(input_dim, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state) for _ in range(n_layers)
        ])
        self.output_proj = nn.Linear(d_model, output_dim)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, input_dim)
        Returns:
            y: (batch, seq_len, output_dim)
        """
        h = self.input_proj(x)
        for layer in self.layers:
            h = layer(h)
        return self.output_proj(h)


# 動作確認
model = MambaModel(input_dim=7, d_model=64, d_state=16, n_layers=2, output_dim=7)
test_input = torch.randn(4, 50, 7)  # batch=4, seq_len=50, joints=7
test_output = model(test_input)
print(f"入力形状:  {test_input.shape}")
print(f"出力形状:  {test_output.shape}")
print(f"総パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

---
## 3. トレーナー

Mambaの学習は、RNNと異なり**系列全体を一度に処理**できます。
Transformerと同様に、全タイムステップの入力を並列に処理し、
全タイムステップの出力を同時に得ることができます。

In [ ]:
class MambaTrainer:
    """Mambaモデルのトレーナー"""
    def __init__(self, model, optimizer, device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device

    def process_epoch(self, x_data, y_data):
        """1エポック分の学習を実行

        Args:
            x_data: 入力系列 (batch, seq_len, input_dim)
            y_data: 目標系列 (batch, seq_len, output_dim)
        Returns:
            loss: 平均損失値
        """
        self.model.train()
        self.optimizer.zero_grad()

        # 全系列を一度に処理（RNNのようなループ不要）
        y_pred = self.model(x_data)
        loss = nn.MSELoss()(y_pred, y_data)

        loss.backward()
        self.optimizer.step()
        return loss.item()

    @torch.no_grad()
    def predict(self, x_data):
        """推論

        Args:
            x_data: 入力系列 (batch, seq_len, input_dim)
        Returns:
            y_pred: 予測系列 (batch, seq_len, output_dim)
        """
        self.model.eval()
        return self.model(x_data)


print("MambaTrainer を定義しました。")

---
## 4. Robomimicデータを使った学習と比較

### 4.1 比較用LSTMモデル

MambaとLSTMの性能を比較するために、同規模のLSTMモデルも定義します。

In [ ]:
class LSTMModel(nn.Module):
    """比較用LSTMモデル"""
    def __init__(self, input_dim, hidden_dim=64, n_layers=2, output_dim=None):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim

        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=n_layers, batch_first=True
        )
        self.output_proj = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.output_proj(h)


print("LSTMModel を定義しました。")
lstm_model = LSTMModel(input_dim=7, hidden_dim=64, n_layers=2, output_dim=7)
print(f"LSTMパラメータ数: {sum(p.numel() for p in lstm_model.parameters()):,}")

### 4.2 データの準備

LeRobot形式のRobomimicデータを読み込みます。
データが利用できない場合はダミーデータを生成します。

In [ ]:
import os

data_path = "/home/ito/b4/data/robomimic_joint_states.npy"

if os.path.exists(data_path):
    raw_data = np.load(data_path, allow_pickle=True)
    print(f"Robomimicデータを読み込みました: {raw_data.shape}")
else:
    # ダミーデータの生成（7自由度ロボットアームの関節角度）
    print("Robomimicデータが見つかりません。ダミーデータを生成します。")
    np.random.seed(42)
    n_episodes = 20
    seq_len = 100
    n_joints = 7

    episodes = []
    for i in range(n_episodes):
        t = np.linspace(0, 4 * np.pi, seq_len)
        # 各関節に異なる周波数・位相のサイン波 + ノイズ
        joint_data = np.zeros((seq_len, n_joints), dtype=np.float32)
        for j in range(n_joints):
            freq = 0.5 + j * 0.3
            phase = i * 0.5 + j * 0.7
            amplitude = 0.5 + 0.3 * np.sin(j)
            joint_data[:, j] = amplitude * np.sin(freq * t + phase)
            joint_data[:, j] += 0.02 * np.random.randn(seq_len)
        episodes.append(joint_data)

    raw_data = np.array(episodes)
    print(f"生成データ形状: {raw_data.shape}  (エピソード数, 系列長, 関節数)")

# 入力: x[t], 目標: x[t+1]
x_all = torch.tensor(raw_data[:, :-1, :], dtype=torch.float32)
y_all = torch.tensor(raw_data[:, 1:, :], dtype=torch.float32)

# 訓練/検証分割
n_train = int(len(x_all) * 0.8)
x_train, x_val = x_all[:n_train], x_all[n_train:]
y_train, y_val = y_all[:n_train], y_all[n_train:]

print(f"訓練データ: x={x_train.shape}, y={y_train.shape}")
print(f"検証データ: x={x_val.shape}, y={y_val.shape}")

### 4.3 学習の実行

MambaモデルとLSTMモデルの両方を同じデータで学習し、損失の推移を比較します。

In [ ]:
n_epochs = 100
input_dim = x_train.shape[-1]

# --- Mambaモデル ---
mamba_model = MambaModel(input_dim=input_dim, d_model=64, d_state=16,
                         n_layers=2, output_dim=input_dim)
mamba_opt = torch.optim.Adam(mamba_model.parameters(), lr=1e-3)
mamba_trainer = MambaTrainer(mamba_model, mamba_opt, device="cpu")

# --- LSTMモデル ---
lstm_model = LSTMModel(input_dim=input_dim, hidden_dim=64,
                       n_layers=2, output_dim=input_dim)
lstm_opt = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

# データをデバイスに転送
x_tr = x_train.to("cpu")
y_tr = y_train.to("cpu")
x_v = x_val.to("cpu")
y_v = y_val.to("cpu")

mamba_losses = []
lstm_losses = []
mamba_val_losses = []
lstm_val_losses = []

print("学習開始...")
for epoch in range(n_epochs):
    # Mamba学習
    m_loss = mamba_trainer.process_epoch(x_tr, y_tr)
    mamba_losses.append(m_loss)

    # LSTM学習
    lstm_model.train()
    lstm_opt.zero_grad()
    lstm_pred = lstm_model(x_tr)
    l_loss = nn.MSELoss()(lstm_pred, y_tr)
    l_loss.backward()
    lstm_opt.step()
    lstm_losses.append(l_loss.item())

    # 検証
    with torch.no_grad():
        mamba_model.eval()
        lstm_model.eval()
        m_val = nn.MSELoss()(mamba_model(x_v), y_v).item()
        l_val = nn.MSELoss()(lstm_model(x_v), y_v).item()
        mamba_val_losses.append(m_val)
        lstm_val_losses.append(l_val)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | "
              f"Mamba train={m_loss:.6f} val={m_val:.6f} | "
              f"LSTM train={l_loss.item():.6f} val={l_val:.6f}")

print("学習完了！")

### 4.4 損失曲線の比較

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 訓練損失
axes[0].plot(mamba_losses, label="Mamba", linewidth=2)
axes[0].plot(lstm_losses, label="LSTM", linewidth=2)
axes[0].set_xlabel("エポック")
axes[0].set_ylabel("MSE損失")
axes[0].set_title("訓練損失の比較")
axes[0].legend()
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

# 検証損失
axes[1].plot(mamba_val_losses, label="Mamba", linewidth=2)
axes[1].plot(lstm_val_losses, label="LSTM", linewidth=2)
axes[1].set_xlabel("エポック")
axes[1].set_ylabel("MSE損失")
axes[1].set_title("検証損失の比較")
axes[1].legend()
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.5 予測軌道の可視化

検証データの1エピソードについて、予測値と真値を比較します。

In [ ]:
# 検証データの最初のエピソードで予測
with torch.no_grad():
    mamba_model.eval()
    lstm_model.eval()
    mamba_pred = mamba_model(x_v[:1]).squeeze(0).numpy()
    lstm_pred_val = lstm_model(x_v[:1]).squeeze(0).numpy()

gt = y_v[0].numpy()

# 4つの関節について可視化
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
joints_to_plot = [0, 2, 4, 6]

for idx, (ax, j) in enumerate(zip(axes.flat, joints_to_plot)):
    ax.plot(gt[:, j], label="真値", linewidth=2, color="black", alpha=0.7)
    ax.plot(mamba_pred[:, j], label="Mamba予測", linewidth=1.5,
            linestyle="--", color="tab:blue")
    ax.plot(lstm_pred_val[:, j], label="LSTM予測", linewidth=1.5,
            linestyle="--", color="tab:orange")
    ax.set_title(f"関節 {j}")
    ax.set_xlabel("タイムステップ")
    ax.set_ylabel("角度")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("予測軌道 vs 真値（検証データ）", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.6 推論速度の比較

異なる系列長でMambaとLSTMの推論時間を比較します。

In [ ]:
seq_lengths = [50, 100, 200, 500, 1000]
mamba_times = []
lstm_times = []

n_repeats = 10

for sl in seq_lengths:
    test_x = torch.randn(1, sl, input_dim)

    # Mamba推論時間
    mamba_model.eval()
    with torch.no_grad():
        # ウォームアップ
        _ = mamba_model(test_x)
        t0 = time.time()
        for _ in range(n_repeats):
            _ = mamba_model(test_x)
        mamba_times.append((time.time() - t0) / n_repeats * 1000)

    # LSTM推論時間
    lstm_model.eval()
    with torch.no_grad():
        _ = lstm_model(test_x)
        t0 = time.time()
        for _ in range(n_repeats):
            _ = lstm_model(test_x)
        lstm_times.append((time.time() - t0) / n_repeats * 1000)

# プロット
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(seq_lengths, mamba_times, "o-", label="Mamba", linewidth=2, markersize=8)
ax.plot(seq_lengths, lstm_times, "s-", label="LSTM", linewidth=2, markersize=8)
ax.set_xlabel("系列長")
ax.set_ylabel("推論時間 (ms)")
ax.set_title("推論速度の比較: Mamba vs LSTM")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 表形式で表示
print(f"{'系列長':>8} | {'Mamba (ms)':>12} | {'LSTM (ms)':>12} | {'比率 (LSTM/Mamba)':>18}")
print("-" * 60)
for sl, mt, lt in zip(seq_lengths, mamba_times, lstm_times):
    ratio = lt / mt if mt > 0 else float('inf')
    print(f"{sl:>8} | {mt:>12.2f} | {lt:>12.2f} | {ratio:>18.2f}")

> **注意:** この教育用実装ではSelectiveSSMの逐次スキャンをPythonのforループで実装しているため、
> 実際のMambaライブラリ（CUDA最適化されたスキャン演算を使用）と比べると大幅に遅くなります。
> 実用では [mamba-ssm](https://github.com/state-spaces/mamba) パッケージの利用を推奨します。

---
## 演習問題

以下の演習で、Mambaの理解を深めましょう。

### 演習1: SelectiveSSMの動作確認

`SelectiveSSM` を使って以下を確認してください：
1. `d_model=16, d_state=8` でSelectiveSSMを作成
2. ランダム入力 `(batch=3, seq_len=20, d_model=16)` を生成
3. 順伝播を実行し、出力形状を確認
4. 各パラメータ（A_log, proj_B, proj_C, proj_delta, D）の形状を表示

**ヒント:**
- `model.named_parameters()` でパラメータ名と値を列挙できます
- 出力形状は入力形状と同じになるはずです

In [ ]:
# SelectiveSSMの作成
d_model = 16
d_state = 8

# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
d_model = 16
d_state = 8

ssm = SelectiveSSM(d_model=d_model, d_state=d_state)

# ランダム入力
x = torch.randn(3, 20, d_model)

# 順伝播
y = ssm(x)

print(f"入力形状: {x.shape}")
print(f"出力形状: {y.shape}")
print(f"形状一致: {x.shape == y.shape}")
print()

# パラメータ一覧
print("パラメータ一覧:")
for name, param in ssm.named_parameters():
    print(f"  {name:20s} -> {param.shape}")

total = sum(p.numel() for p in ssm.parameters())
print(f"\n総パラメータ数: {total:,}")
```

</details>

### 演習2: MambaModelを使った関節角度予測学習

以下の手順でMambaModelを学習してください：
1. `d_model=32, d_state=8, n_layers=3` でMambaModelを作成
2. 学習率 `5e-4` のAdamオプティマイザを設定
3. `MambaTrainer` を使って150エポック学習
4. 損失曲線をプロットし、最終的な訓練・検証損失を表示

**ヒント:**
- `x_train, y_train, x_val, y_val` は既に定義済みです
- 20エポックごとに損失を表示すると進捗が分かりやすいです

In [ ]:
# モデル作成
# ここにコードを書いてください

# 学習ループ
# ここにコードを書いてください

# 損失曲線のプロット
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
# モデル作成
my_model = MambaModel(input_dim=input_dim, d_model=32, d_state=8,
                      n_layers=3, output_dim=input_dim)
my_opt = torch.optim.Adam(my_model.parameters(), lr=5e-4)
my_trainer = MambaTrainer(my_model, my_opt, device="cpu")

print(f"パラメータ数: {sum(p.numel() for p in my_model.parameters()):,}")

# 学習ループ
train_losses = []
val_losses = []

for epoch in range(150):
    loss = my_trainer.process_epoch(x_train, y_train)
    train_losses.append(loss)

    with torch.no_grad():
        my_model.eval()
        v_loss = nn.MSELoss()(my_model(x_val), y_val).item()
        val_losses.append(v_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/150 | train={loss:.6f} | val={v_loss:.6f}")

# 損失曲線のプロット
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="訓練損失")
plt.plot(val_losses, label="検証損失")
plt.xlabel("エポック")
plt.ylabel("MSE損失")
plt.title("MambaModel 学習曲線 (d_model=32, n_layers=3)")
plt.legend()
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

print(f"最終訓練損失: {train_losses[-1]:.6f}")
print(f"最終検証損失: {val_losses[-1]:.6f}")
```

</details>

### 演習3: MambaとLSTMの推論速度比較

異なる系列長（50, 100, 200, 500, 1000, 2000）で両モデルの推論速度を比較してください：
1. 各系列長で20回推論を繰り返し、平均時間を計測
2. 系列長に対する推論時間のグラフを作成
3. 系列長が長くなるにつれてのスケーリング特性を考察

**ヒント:**
- `time.time()` で計測できます
- ウォームアップとして事前に1回推論を実行しましょう
- `torch.no_grad()` を忘れずに

In [ ]:
seq_lengths = [50, 100, 200, 500, 1000, 2000]
n_repeats = 20

# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
seq_lengths = [50, 100, 200, 500, 1000, 2000]
n_repeats = 20

# 比較用モデル
m_model = MambaModel(input_dim=7, d_model=64, d_state=16, n_layers=2, output_dim=7)
l_model = LSTMModel(input_dim=7, hidden_dim=64, n_layers=2, output_dim=7)

m_model.eval()
l_model.eval()

m_times = []
l_times = []

for sl in seq_lengths:
    x = torch.randn(1, sl, 7)

    # Mamba
    with torch.no_grad():
        _ = m_model(x)  # ウォームアップ
        t0 = time.time()
        for _ in range(n_repeats):
            _ = m_model(x)
        m_times.append((time.time() - t0) / n_repeats * 1000)

    # LSTM
    with torch.no_grad():
        _ = l_model(x)
        t0 = time.time()
        for _ in range(n_repeats):
            _ = l_model(x)
        l_times.append((time.time() - t0) / n_repeats * 1000)

    print(f"seq_len={sl:5d} | Mamba: {m_times[-1]:8.2f} ms | LSTM: {l_times[-1]:8.2f} ms")

# プロット
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, m_times, "o-", label="Mamba", linewidth=2)
axes[0].plot(seq_lengths, l_times, "s-", label="LSTM", linewidth=2)
axes[0].set_xlabel("系列長")
axes[0].set_ylabel("推論時間 (ms)")
axes[0].set_title("推論時間の比較")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 対数スケール
axes[1].plot(seq_lengths, m_times, "o-", label="Mamba", linewidth=2)
axes[1].plot(seq_lengths, l_times, "s-", label="LSTM", linewidth=2)
axes[1].set_xlabel("系列長")
axes[1].set_ylabel("推論時間 (ms)")
axes[1].set_title("推論時間の比較（対数スケール）")
axes[1].legend()
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
```

</details>

---
## まとめ

| 項目 | 内容 |
|------|------|
| SSMの基本 | 連続時間 $x'=Ax+Bu$ を離散化して $x_k = \bar{A}x_{k-1} + \bar{B}u_k$ |
| Mambaの特徴 | $B, C, \Delta$ が入力依存（選択的SSM） |
| 計算量 | O(N) --- Transformerの O(N^2) に対して線形 |
| 学習方法 | 全系列を一度に処理（RNNと異なり並列化可能） |
| 推論 | RNNのように逐次処理が可能（固定メモリ） |

### 次のステップ

- `mamba-ssm` パッケージを使った高速実装への移行
- 画像特徴量との統合（マルチモーダルMamba）
- より長い系列での性能検証